# Notas — Aula 8: Dataclasses e enums

Marco: `Posicao` e `Comando`, escritos à mão na Aula 3 (`__init__`/`__repr__`/`__eq__`
um por um), viram `@dataclass` em poucas linhas; uma classe nova, `Leitura` (registro
de sensor), já nasce como dataclass. Depois, `Robo.direcao` — hoje só uma string
validada por um `if` — vira `Enum` (`Direcao`), cujo valor é o próprio vetor de
movimento e cujos métodos (`virar_esquerda`/`virar_direita`) substituem os
dicionários `GIRAR_ESQ`/`GIRAR_DIR`.

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**) — cada
> seção depende da anterior.

In [1]:
LADO_GRADE = 10

## `@dataclass` — menos código para guardar dados

`@dataclass` lê as **anotações de tipo** dos atributos e gera `__init__`, `__repr__` e
`__eq__` sozinho — o mesmo trio que `Posicao` e `Comando` ganharam à mão na Aula 3.
Um atributo **sem** `: tipo` não vira campo — some do `__init__` e do `__repr__`, sem
aviso. No robô, usamos isso para `Posicao` e `Comando`: menos código, mesma promessa.

In [2]:
from dataclasses import dataclass


@dataclass
class Posicao:
    x: int
    y: int


p1 = Posicao(3, 4)
p2 = Posicao(3, 4)
print(p1)
print(p1 == p2)
print(p1 == (3, 4))

Posicao(x=3, y=4)
True
False


In [3]:
@dataclass
class Comando:
    acao: str
    valor: int | str


c1 = Comando("AVANCAR", 3)
c2 = Comando("AVANCAR", 3)
print(c1)
print(c1 == c2)

Comando(acao='AVANCAR', valor=3)
True


### Sua vez

Complete a anotação de tipo que falta em `ComandoIncompleto`: `valor` precisa virar
`valor: int = 0` para entrar no `__init__`/`__repr__` gerados.

*Dica: mesmo problema do `PosicaoQuebrada` visto em sala — falta o `: tipo`.*

In [4]:
@dataclass
class ComandoIncompleto:
    acao: str
    valor = 0        # TODO: adicione a anotação de tipo que falta (": int")


c = ComandoIncompleto("AVANCAR")
print(c)
print(c.valor)

ComandoIncompleto(acao='AVANCAR')
0


## `frozen=True` — trava reatribuição

`frozen=True` faz qualquer tentativa de reatribuir um campo levantar
`FrozenInstanceError` (uma subclasse de `AttributeError`) — o mesmo tipo de erro de
tentar escrever numa `@property` só leitura. Faz sentido em registros que representam
um instante que já passou, como `Posicao` guardada em `trajetoria`.

In [5]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Posicao:
    x: int
    y: int


p = Posicao(3, 4)
try:
    p.x = 99
except Exception as erro:
    print(f"{type(erro).__name__}: {erro}")

FrozenInstanceError: cannot assign to field 'x'


In [6]:
@dataclass(frozen=True)
class Comando:
    acao: str
    valor: int | str


c = Comando("AVANCAR", 3)
try:
    c.valor = 5
except Exception as erro:
    print(f"{type(erro).__name__}: {erro}")

FrozenInstanceError: cannot assign to field 'valor'


### Sua vez

Complete o decorador de `Sensor` com `frozen=True`, para que reatribuir `nome`
depois de criado levante `FrozenInstanceError`.

*Dica: `@dataclass(frozen=True)`, igual em `Posicao`/`Comando` acima.*

In [7]:
@dataclass  # TODO: acrescente frozen=True
class Sensor:
    nome: str
    alcance: int


s = Sensor("frente", 5)
try:
    s.nome = "trás"
    print("reatribuiu sem erro:", s.nome)
except Exception as erro:
    print(f"{type(erro).__name__}: {erro}")

reatribuiu sem erro: trás


## `field(default_factory=...)` — evita valor default compartilhado

Mesma armadilha do argumento default mutável (`obstaculos={}` no `Robo`, lá na Aula
1), agora dentro de uma dataclass: `campo: list = []` nem chega a rodar —
`@dataclass` recusa com `ValueError` na hora de definir a classe. O conserto é
`field(default_factory=list)`, que cria uma lista **nova** para cada instância.

In [8]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Leitura:
    posicao: tuple
    obstaculos_proximos: list = field(default_factory=list)


l1 = Leitura((0, 0))
l2 = Leitura((0, 0))
print(l1)
print(l1.obstaculos_proximos is l2.obstaculos_proximos)

Leitura(posicao=(0, 0), obstaculos_proximos=[])
False


In [9]:
try:
    @dataclass(frozen=True)
    class LeituraRuim:
        posicao: tuple
        obstaculos_proximos: list = []      # ERRO — mesmo problema de sempre
except ValueError as erro:
    print(f"ValueError: {erro}")

ValueError: mutable default <class 'list'> for field obstaculos_proximos is not allowed: use default_factory


### Sua vez

Complete `HistoricoComandos`: troque `comandos: list = []` (que nem chega a rodar)
por `field(default_factory=list)`.

*Dica: mesmo conserto de `Leitura.obstaculos_proximos` acima.*

In [10]:
try:
    @dataclass
    class HistoricoComandos:
        comandos: list = []      # TODO: troque por field(default_factory=list)

    h1 = HistoricoComandos()
    h2 = HistoricoComandos()
    print(h1.comandos is h2.comandos)
except ValueError as erro:
    print(f"ValueError: {erro}")

ValueError: mutable default <class 'list'> for field comandos is not allowed: use default_factory


## A armadilha: `frozen=True` não protege o conteúdo de um campo mutável

`frozen=True` impede **reatribuir** um campo (`l1.posicao = ...`) — não impede
**mutar o objeto que já está lá dentro**. Se o campo é uma lista, `frozen` protege só
o rótulo, não o conteúdo: `l1.obstaculos_proximos.append(...)` funciona, calado.
Imutabilidade de `frozen` é **rasa**, não profunda.

In [11]:
l1 = Leitura((2, 2), [(3, 2)])
try:
    l1.posicao = (0, 0)
except Exception as erro:
    print(f"{type(erro).__name__}: {erro}")

l1.obstaculos_proximos.append((5, 5))
print(l1)

FrozenInstanceError: cannot assign to field 'posicao'
Leitura(posicao=(2, 2), obstaculos_proximos=[(3, 2), (5, 5)])


In [12]:
@dataclass(frozen=True)
class Rota:
    pontos: list = field(default_factory=list)


rota = Rota()
rota.pontos.append((0, 0))
rota.pontos.append((1, 0))
print(rota)

Rota(pontos=[(0, 0), (1, 0)])


### Sua vez

Complete `adicionar_ponto(rota, ponto)`: acrescente `ponto` a `rota.pontos` — mesmo
que `Rota` seja `frozen`, a lista de dentro continua mutável.

*Dica: uma linha — `rota.pontos.append(ponto)`.*

In [13]:
def adicionar_ponto(rota, ponto):
    # TODO: acrescente ponto a rota.pontos
    pass


rota2 = Rota()
adicionar_ponto(rota2, (3, 3))
print(rota2)

Rota(pontos=[])


## `Enum` — um conjunto fechado de valores válidos

Sem `Enum`, `direcao` é só uma string validada por um `if` — um typo (`"Leste"` com L
maiúsculo) só é pego se alguém lembrar de checar. `Enum` fecha o conjunto de valores
**de verdade**: `Direcao.LESTE` não é igual a `"LESTE"`, e o **valor** de cada membro
pode carregar dado útil — aqui, o próprio vetor de movimento, substituindo o
dicionário `DELTAS`.

In [14]:
from enum import Enum, auto


class Prioridade(Enum):
    BAIXA = auto()
    MEDIA = auto()
    ALTA = auto()


p = Prioridade.ALTA
print(p)
print(p.name, p.value)
print(Prioridade.ALTA == Prioridade.ALTA)
print(Prioridade.ALTA == "ALTA")

Prioridade.ALTA
ALTA 3
True
False


In [15]:
class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


print(Direcao.LESTE)
print(Direcao.LESTE.value)
dx, dy = Direcao.LESTE.value
print(dx, dy)
print(Direcao((1, 0)))

Direcao.LESTE
(1, 0)
1 0
Direcao.LESTE


### Sua vez

Complete `eh_horizontal(self)` em `Direcao`: devolva `True` se o vetor tiver `dx != 0`
(ou seja, `LESTE`/`OESTE`), `False` senão.

*Dica: `dx, dy = self.value`, depois compare `dx != 0`.*

In [16]:
class DirecaoInfo(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)

    def eh_horizontal(self):
        # TODO: devolva True se dx != 0 (LESTE/OESTE), False senão
        pass


print(DirecaoInfo.LESTE.eh_horizontal())
print(DirecaoInfo.NORTE.eh_horizontal())

None
None


## Métodos no Enum, integrados no `Robo`

`virar_esquerda`/`virar_direita` substituem os dicionários `GIRAR_ESQ`/`GIRAR_DIR`: o
enum sabe girar a si mesmo. Referenciar `Direcao.LESTE` **dentro** de um método da
própria classe funciona porque o método só roda quando é chamado — nesse momento a
classe já existe inteira. No robô, `self.direcao.value` substitui `DELTAS[...]` e
`self.direcao.virar_esquerda()` substitui `GIRAR_ESQ[...]`.

In [17]:
class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)

    def virar_esquerda(self):
        ordem = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]
        return ordem[(ordem.index(self) + 1) % 4]

    def virar_direita(self):
        ordem = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]
        return ordem[(ordem.index(self) - 1) % 4]


print(Direcao.LESTE.virar_esquerda())
print(Direcao.LESTE.virar_direita())

Direcao.NORTE
Direcao.SUL


In [18]:
class Robo:
    LADO_GRADE = 10

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao

    def girar(self, lado):
        if lado == "ESQ":
            self.direcao = self.direcao.virar_esquerda()
        elif lado == "DIR":
            self.direcao = self.direcao.virar_direita()


robo1 = Robo("Wall-E")
robo1.girar("ESQ")
print(robo1.direcao)

Direcao.NORTE


### Sua vez

Complete `avancar(self)`: descubra `dx, dy` a partir de `self.direcao.value` e some
em `self.x`/`self.y` (sem checar limites da grade — só o movimento).

*Dica: `dx, dy = self.direcao.value`, depois `self.x += dx` e `self.y += dy`.*

In [19]:
class RoboMini:
    def __init__(self, direcao=Direcao.LESTE, x=0, y=0):
        self.direcao = direcao
        self.x = x
        self.y = y

    def avancar(self):
        # TODO: dx, dy = self.direcao.value; some dx em self.x e dy em self.y
        pass


r = RoboMini(direcao=Direcao.NORTE)
r.avancar()
print(r.x, r.y)

0 0


## Para aprofundar

- `dataclasses` (visão geral) — documentação oficial: https://docs.python.org/3/library/dataclasses.html
- `frozen`, `field`, `default_factory` — documentação oficial: https://docs.python.org/3/library/dataclasses.html#dataclasses.field · Real Python — Data Classes: https://realpython.com/python-data-classes/
- `enum` (visão geral) — documentação oficial: https://docs.python.org/3/library/enum.html
- Enum com métodos e `auto()` — Real Python: https://realpython.com/python-enum/